In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS felipe_learning_db;
USE felipe_learning_db;


In [0]:
%fs ls dbfs:/databricks-datasets/amazon/data20K

In [0]:
%sql
SELECT * FROM parquet.`dbfs:/databricks-datasets/amazon/data20K/part-r-00000-112e73de-1ab1-447b-b167-0919dd731adf.gz.parquet`

In [0]:
%sql
SELECT * FROM read_files('dbfs:/databricks-datasets/amazon/data20K/part-r-00000-112e73de-1ab1-447b-b167-0919dd731adf.gz.parquet')

In [0]:
%sql
DROP TABLE IF EXISTS bronze_data20k_ctas_rf;
    
CREATE TABLE IF NOT EXISTS bronze_data20k_ctas_rf
AS
SELECT * FROM read_files('dbfs:/databricks-datasets/amazon/data20K/part-r-00000-112e73de-1ab1-447b-b167-0919dd731adf.gz.parquet');

SELECT * FROM bronze_data20k_ctas_rf LIMIT 10;

In [0]:
%sql
DESCRIBE TABLE EXTENDED bronze_data20k_ctas_rf; -- ctas: copy table as and _rf: read_files 


## Python Ingestion

In [0]:
df = (spark
      .read
      .format("parquet")
      .load("dbfs:/databricks-datasets/amazon/data20K/part-r-00000-112e73de-1ab1-447b-b167-0919dd731adf.gz.parquet")
      )

(df
 .write
 .mode("overwrite")
 .saveAsTable(f"workspace.felipe_learning_db.bronze_data20K_python")
)

data20K_bronze_table = spark.table("workspace.felipe_learning_db.bronze_data20K_python")
data20K_bronze_table.display()

## Incremental Data Ingestion with COPY INTO

In [0]:
%sql
DROP TABLE IF EXISTS bronze_data20k_ci; -- ci: COPY INTO

CREATE TABLE IF NOT EXISTS bronze_data20k_ci (
    rating double       
);

COPY INTO bronze_data20k_ci
FROM 'dbfs:/databricks-datasets/amazon/data20K'
FILEFORMAT = PARQUET;

In [0]:
%sql
COPY INTO bronze_data20K_ci
FROM 'dbfs:/databricks-datasets/amazon/data20K'
FILEFORMAT = parquet
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
-- select * from bronze_data20k_ctas;
    
select * from bronze_data20k_ci limit 10;


##Preemptively Handling Schema Evolution

In [0]:
%sql
DROP TABLE IF EXISTS bronze_data20k_ci_no_schema;

CREATE TABLE bronze_data20k_ci_no_schema;

COPY INTO bronze_data20k_ci_no_schema
FROM 'dbfs:/databricks-datasets/amazon/data20K'
FILEFORMAT = PARQUET
COPY_OPTIONS("mergeSchema" = "true");




## Idempotency

In [0]:
%sql
COPY INTO bronze_data20k_ci_no_schema
FROM 'dbfs:/databricks-datasets/amazon/data20K'
FILEFORMAT = parquet
COPY_OPTIONS ('mergeSchema' = 'true');


In [0]:
%sql
COPY INTO tabela_teste_copy_into
FROM 'dbfs:/databricks-datasets/flights/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true');

In [0]:
%sql
select current_catalog(), current_schema();
